# Fine-tune a small local "Jarvis" model (Google Colab, free GPU)

This notebook LoRA fine-tunes a small open-source chat model (TinyLlama-1.1B-Chat) on your own
Jarvis-style Q&A data, so it picks up a particular persona, tone, or domain knowledge.

**Important honesty note:** this does NOT create a model as capable as Gemini/GPT-4. A 1B-parameter
model fine-tuned for a few hundred steps on Colab's free tier will mimic a *style* and can memorize
small, specific facts you give it -- it will not reason as well as a large hosted model. In the
companion project, Gemini is used for day-to-day answers, and this fine-tuned model is optional/for
experimentation. If you want it as the main brain, set `USE_LOCAL_FINETUNED_MODEL=true` in the
backend's `.env` after downloading the model folder produced here.

**Steps:**
1. Runtime -> Change runtime type -> GPU (T4 is fine, free tier).
2. Run all cells top to bottom.
3. Edit the `training_data` list (or upload your own `.jsonl`) with your own Q&A pairs.
4. At the end, download `jarvis_finetuned_model.zip` and unzip it into your project as
   `backend/finetuned_jarvis_model/`.

In [ ]:
!pip install -q transformers==4.44.2 peft==0.13.0 accelerate==0.34.2 bitsandbytes==0.43.3 datasets==2.21.0 trl==0.9.6

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import Dataset

print('GPU available:', torch.cuda.is_available())

## 1. Your training data

Replace the examples below with your own persona / domain Q&A pairs. More examples (50-500+)
and consistent formatting give better results. You can also upload a `data.jsonl` file with
one `{"instruction": ..., "response": ...}` object per line and load it instead -- see the
commented-out loader cell further down.

In [ ]:
training_data = [
    {"instruction": "Who are you?", "response": "I am Jarvis, your personal assistant. I'm here to help, sir/ma'am."},
    {"instruction": "What can you do?", "response": "I can answer questions, manage information you give me, and assist with day-to-day tasks -- all running locally on your machine."},
    {"instruction": "Good morning Jarvis", "response": "Good morning. Systems are online and ready whenever you are."},
    {"instruction": "Thank you", "response": "Always a pleasure. Let me know if there's anything else you need."}
    # Add many more examples here...
]

def to_text(example):
    return {
        "text": f"<|system|>\nYou are JARVIS, a calm and precise personal AI assistant.\n<|user|>\n{example['instruction']}\n<|assistant|>\n{example['response']}"
    }

dataset = Dataset.from_list(training_data).map(to_text)
dataset[0]

### Optional: load from an uploaded .jsonl file instead
Uncomment and run this cell if you'd rather upload a dataset file (Files icon on the left -> upload).

In [ ]:
# import json
# rows = [json.loads(line) for line in open('data.jsonl')]
# dataset = Dataset.from_list(rows).map(to_text)

## 2. Load base model + tokenizer

In [ ]:
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

## 3. LoRA config + training

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

training_args = TrainingArguments(
    output_dir="./jarvis_lora_checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to=[]
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512
)

trainer.train()

## 4. Merge LoRA adapters into the base model and save

In [ ]:
merged_model = trainer.model.merge_and_unload()

save_path = "./jarvis_finetuned_model"
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print('Saved to', save_path)

## 5. Zip and download
Download the zip, unzip it, and place the folder at `backend/finetuned_jarvis_model/` in your
local project. Then set `USE_LOCAL_FINETUNED_MODEL=true` and
`LOCAL_MODEL_PATH=./finetuned_jarvis_model` in `backend/.env` if you want the backend to use it
instead of Gemini.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("jarvis_finetuned_model", "zip", save_path)
files.download("jarvis_finetuned_model.zip")